# ReviewSense PT-BR — 03 · BERTimbau: piloto e fine-tuning

Este notebook **inspeciona resultados salvos**; não inicia ou repete treinamento.
As decisões computacionais são separadas da seleção de época por F1 macro.
O teste final permanece reservado. Pesos, tokens individuais e estados de retomada
ficam locais. Veja `docs/FASE3B_BERTIMBAU.md` para os comandos de execução.

In [ ]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

candidates = [Path(os.environ["REVIEWSENSE_REPO"])] if "REVIEWSENSE_REPO" in os.environ else [
    Path.cwd(), Path.cwd().parent
]
ROOT = next((path.resolve() for path in candidates
             if (path / "src/reviewsense/phase3b.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Defina REVIEWSENSE_REPO com a raiz do projeto.")
os.chdir(ROOT)
from reviewsense.evaluate import plot_confusion_matrix

lengths = json.loads((ROOT / "reports/phase3b_token_lengths.json").read_text())
pilot = json.loads((ROOT / "reports/phase3b_gpu_pilot.json").read_text())

## Comprimentos completos antes do truncamento

Todos os textos de treino e validação, incluindo CLS/SEP. Percentis lineares;
porcentagens estritamente acima de 128/256. Nenhum dado de teste foi aberto.

In [ ]:
display(pd.DataFrame({
    name: {**info["percentiles"], "acima_128_pct": info["exceeds"]["128"]["percent"],
           "acima_256_pct": info["exceeds"]["256"]["percent"], "amostras": info["samples"]}
    for name, info in lengths["lengths"].items()
}).T)

## Piloto computacional

As mesmas avaliações representativas e longas, com duas repetições por regime.
A decisão usa somente estabilidade e margem mínima de 1 GiB; não usa F1 ou loss
mínima. A escala FP16 foi adaptada por overflow numérico, sem alterar LR.
O consumo do driver inclui outros processos e é amostrado por NVML a cada 50 ms.

In [ ]:
display(pd.DataFrame([
    {"variante": row["variant_id"],
     "ex_s_representativos": row["regimes"]["representative"]["median_examples_per_second"],
     "ex_s_longos": row["regimes"]["long_stress"]["median_examples_per_second"],
     "pico_alocado_GiB": row["pytorch"]["peak_allocated_bytes"] / 2**30,
     "pico_reservado_GiB": row["pytorch"]["peak_reserved_bytes"] / 2**30,
     "pico_driver_GiB": row["driver"]["peak_used_bytes_sampled"] / 2**30,
     "min_livre_driver_GiB": row["driver"]["min_free_bytes"] / 2**30,
     "estavel": row["numerically_stable"]}
    for row in pilot["validated_scaler_round"]
]))
display(pd.Series(pilot["acceptance"]["selected_config"]))
display(pd.Series(pilot["acceptance"]["resume_proof"]))

## Treinamento completo e escolha de checkpoint

Até três épocas; F1 macro na validação inteira seleciona o checkpoint. Não há nova
busca de hiperparâmetros por resultados do piloto. As métricas de validação sofrem
viés de seleção e não substituem uma avaliação independente final.

In [ ]:
report = ROOT / "reports/bertimbau_phase3b_metrics.json"
if not report.exists():
    report = ROOT / "models/bertimbau/phase3b-full-v3/results.json"
results = json.loads(report.read_text()) if report.exists() else None
if results is None:
    print("Treinamento ainda não concluído; consulte progress.json na execução local.")
else:
    display(pd.DataFrame([
        {"epoca": row["epoch"], "update": row["global_step"],
         **{key: row["metrics"][key] for key in ["f1_macro", "precision_macro", "recall_macro", "accuracy"]}}
        for row in results["validation_history"]
    ]))
    print("Checkpoint selecionado:", results["best_checkpoint"])
    print("F1 macro selecionado:", results["selected_validation"]["f1_macro"])

In [ ]:
if results is not None:
    history = pd.DataFrame([
        {"epoca": row["epoch"], "f1_macro": row["metrics"]["f1_macro"]}
        for row in results["validation_history"]
    ])
    ax = history.plot(x="epoca", y="f1_macro", marker="o", legend=False)
    ax.set(title="F1 macro de validação por época", xlabel="Época", ylabel="F1 macro")
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame(results["selected_validation"]["per_class"]).T)
    figure = plot_confusion_matrix(results["selected_validation"], title="BERTimbau — validação")
    plt.show()
    plt.close(figure)

## Retomada e limitações

`best_model/` contém pesos e tokenizer para inferência. A retomada usa
`checkpoint-N/`, com optimizer, scheduler, scaler, RNG e posição dos dados.
O pipeline guarda o melhor checkpoint e o mais recente. Nunca se retoma de um
arquivo desconhecido ou apenas dos pesos do melhor modelo.

A prova prévia em GPU verificou igualdade de pesos e estados ao interromper no
update 2 e retomar até 4 em outro processo. O treino completo também mantém
checkpoints a cada 500 updates e ao fim das épocas.

As estrelas são supervisão indireta; o domínio é comércio eletrônico de 2018.
O piloto curto não mede aquecimento sustentado. Truncar preserva apenas parte dos
textos longos. O teste final não foi usado; não há promessa de generalização.